# BIOL 363 — Nuclear Receptor Cofactor Geometry Pipeline

**Student B: Geometry Infrastructure**

This notebook is the cleaned, reproducible version of the exploratory analysis. It contains the validated workflow only; debugging cells, one-off checks, obsolete intermediate methods, and duplicated calculations were removed.

## What this notebook does

1. Audits the Student A handoff before geometry.
2. Maps every receptor chain to **canonical UniProt numbering**.
3. Runs/resumes **P2Rank** and selects the correct receptor-chain pocket.
4. Maps P2Rank pocket residues to canonical numbering.
5. Computes a **SASA-weighted pocket centroid**.
6. Validates centroids on up to five structures per receptor using scripted PyMOL measurements plus manual review of outliers.
7. Builds canonical **H12 reference candidates** and prepares the H12 PCA workflow.
8. Saves reusable outputs to CSV/Parquet so downstream students do not recompute them.

## Current validated results from the completed run

- Geometry-ready dataset: **1,167 receptor-chain units**, **708 unique PDBs**, **28 receptor types**.
- Pre-geometry canonical-numbering gate: **1,167/1,167 PASS**; **0 GR numbering failures**.
- Final P2Rank pocket table: **1,167/1,167 PASS**; no chain-dominance or <80% canonical-mapping failures.
- Centroid validation set: **121 cases across 28 receptors**, all ultimately PASS after scripted PyMOL QC and manual review of flagged cases.
- Full centroid batch: **1,167/1,167 computed**; 1,163 passed automatic QC and 4 shift outliers passed manual PyMOL review.
- H12: candidate/reference QC is in progress; full H12 scaling must not be claimed complete until the H12 validation gate passes.

## 0. Reproducibility rules

The pipeline follows four rules requested by the supervisor:

- **Canonical UniProt numbering before geometry.** Deposited construct numbering is never treated as canonical numbering.
- **Parameterised processing.** The same functions run across all receptors; there is no manual family-by-family script editing.
- **Progress bars + parallelisation + error handling** for long batch operations.
- **Save outputs to disk** (CSV/Parquet/JSON) and reuse them instead of recomputing.

### Repository layout

This notebook is designed for the team repository:

```text
BIOL363-NR-Cofactor-Analysis/
├── data/
│   ├── snapshot/                 # supplied Student A / previous-cohort CSVs
│   └── generated/                # ignored runtime outputs
├── notebooks/
├── scripts/
├── README.md
└── requirements.txt
```

Downloaded mmCIF files, P2Rank raw output, and runtime products belong under `data/generated/` and are ignored by Git.

In [ ]:
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import os
import re
import subprocess
import time

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

from Bio.Align import PairwiseAligner
from Bio.PDB import MMCIFParser
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from Bio.PDB.Polypeptide import is_aa
from Bio.PDB.SASA import ShrakeRupley
from Bio.SeqUtils import seq1

def find_repo_root(start=None):
    start=Path(start or Path.cwd()).resolve()
    for p in [start,*start.parents]:
        if (p/'README.md').exists() and (p/'scripts').exists() and (p/'data').exists():
            return p
    # local legacy layout fallback
    if (start/'input').exists():
        return start
    raise FileNotFoundError(
        'Could not find repository root. Run from the repository or set ROOT manually.'
    )

ROOT=find_repo_root()

SNAPSHOT=ROOT/'data'/'snapshot'
if not SNAPSHOT.exists():
    SNAPSHOT=ROOT/'input'  # legacy local layout

GENERATED=ROOT/'data'/'generated'
STRUCTURES=GENERATED/'structures'
PROCESSED=GENERATED/'geometry'
NUMBERING=GENERATED/'numbering'
P2OUT=GENERATED/'p2rank_out'
VALIDATION=GENERATED/'validation'

for p in [STRUCTURES,PROCESSED,NUMBERING,P2OUT,VALIDATION]:
    p.mkdir(parents=True,exist_ok=True)

# P2Rank is an external Java program and should not be committed to Git.
p2rank_home=os.environ.get('P2RANK_HOME')
if p2rank_home:
    P2RANK=Path(p2rank_home)
else:
    candidates=[
        ROOT/'p2rank_2.4.2',
        ROOT.parent/'p2rank_2.4.2'
    ]
    P2RANK=next((p for p in candidates if p.exists()),candidates[0])

PRANK=P2RANK/('prank.bat' if os.name=='nt' else 'prank')

N_WORKERS=min(8,max(1,(os.cpu_count() or 4)-1))
P2RANK_WORKERS=min(2,N_WORKERS)

print('Repository root:',ROOT)
print('Input snapshot:',SNAPSHOT)
print('Generated output:',GENERATED)
print('Workers:',N_WORKERS)
print('P2Rank:',PRANK)
print('P2Rank executable exists:',PRANK.exists())

## 1. Nuclear-receptor metadata

The supervisor-provided metadata is defined before any curation or geometry. `uniprot_id` is the stable technical identifier. Common names (e.g. PPARγ, RXRα) are presentation labels only.

In [ ]:
NR_METADATA = [

# =========================================================================
# STEROID HORMONE RECEPTORS
# =========================================================================

{
"uniprot_id": "P10275",
"nr_code": "NR3C4",
"common_name": "AR",
"search_terms": ["Androgen receptor"],
"group": "Steroid hormone receptors"
},

{
"uniprot_id": "P03372",
"nr_code": "NR3A1",
"common_name": "ERα",
"search_terms": ["Estrogen receptor"],
"group": "Steroid hormone receptors"
},

{
"uniprot_id": "Q92731",
"nr_code": "NR3A2",
"common_name": "ERβ",
"search_terms": ["Estrogen receptor beta"],
"group": "Steroid hormone receptors"
},

{
"uniprot_id": "P04150",
"nr_code": "NR3C1",
"common_name": "GR",
"search_terms": ["Glucocorticoid receptor"],
"group": "Steroid hormone receptors"
},

{
"uniprot_id": "P08235",
"nr_code": "NR3C2",
"common_name": "MR",
"search_terms": ["Mineralocorticoid receptor"],
"group": "Steroid hormone receptors"
},

{
"uniprot_id": "P06401",
"nr_code": "NR3C3",
"common_name": "PR",
"search_terms": ["Progesterone receptor"],
"group": "Steroid hormone receptors"
},

{
"uniprot_id": "P10276",
"nr_code": "NR1B1",
"common_name": "RARα",
"search_terms": ["Retinoic acid receptor alpha"],
"group": "Steroid hormone receptors"
},

{
"uniprot_id": "P10826",
"nr_code": "NR1B2",
"common_name": "RARβ",
"search_terms": ["Retinoic acid receptor beta"],
"group": "Steroid hormone receptors"
},

{
"uniprot_id": "P13631",
"nr_code": "NR1B3",
"common_name": "RARγ",
"search_terms": ["Retinoic acid receptor gamma"],
"group": "Steroid hormone receptors"
},

{
"uniprot_id": "P10827",
"nr_code": "NR1A1",
"common_name": "TRα",
"search_terms": ["Thyroid hormone receptor alpha"],
"group": "Steroid hormone receptors"
},

{
"uniprot_id": "P10828",
"nr_code": "NR1A2",
"common_name": "TRβ",
"search_terms": ["Thyroid hormone receptor beta"],
"group": "Steroid hormone receptors"
},

{
"uniprot_id": "P11473",
"nr_code": "NR1I1",
"common_name": "VDR",
"search_terms": [
"Vitamin D3 receptor",
"Vitamin D3 receptor A",
"Vitamin D receptor"
],
"group": "Steroid hormone receptors"
},


# =========================================================================
# NUTRITIONAL SENSORS
# =========================================================================

{
"uniprot_id": "Q14994",
"nr_code": "NR1I3",
"common_name": "CAR",
"search_terms": [
"Nuclear receptor subfamily 1 group I member 3"
],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P11474",
"nr_code": "NR3B1",
"common_name": "ERRα",
"search_terms": ["Steroid hormone receptor ERR1"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "O95718",
"nr_code": "NR3B2",
"common_name": "ERRβ",
"search_terms": ["Steroid hormone receptor ERR2"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P62508",
"nr_code": "NR3B3",
"common_name": "ERRγ",
"search_terms": ["Estrogen-related receptor gamma"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "Q96RI1",
"nr_code": "NR1H4",
"common_name": "FXR",
"search_terms": ["Bile acid receptor"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P41235",
"nr_code": "NR2A1",
"common_name": "HNF4α",
"search_terms": ["Hepatocyte nuclear factor 4-alpha"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "Q14541",
"nr_code": "NR2A2",
"common_name": "HNF4γ",
"search_terms": ["Hepatocyte nuclear factor 4-gamma"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "O00482",
"nr_code": "NR5A2",
"common_name": "LRH-1",
"search_terms": [
"Nuclear receptor subfamily 5 group A member 2"
],
"group": "Nutritional sensors"
},

{
"uniprot_id": "Q13133",
"nr_code": "NR1H3",
"common_name": "LXRα",
"search_terms": ["Oxysterols receptor LXR-alpha"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P55055",
"nr_code": "NR1H2",
"common_name": "LXRβ",
"search_terms": ["OXYSTEROLS RECEPTOR LXR-BETA"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P22736",
"nr_code": "NR4A1",
"common_name": "NUR77",
"search_terms": [
"Nuclear receptor subfamily 4 group A member 1"
],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P43354",
"nr_code": "NR4A2",
"common_name": "NURR1",
"search_terms": [
"Nuclear receptor subfamily 4 group A member 2"
],
"group": "Nutritional sensors"
},

{
"uniprot_id": "Q07869",
"nr_code": "NR1C1",
"common_name": "PPARα",
"search_terms": [
"Peroxisome proliferator-activated receptor alpha"
],
"group": "Nutritional sensors"
},

{
"uniprot_id": "Q03181",
"nr_code": "NR1C2",
"common_name": "PPARδ",
"search_terms": [
"Peroxisome proliferator-activated receptor delta"
],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P37231",
"nr_code": "NR1C3",
"common_name": "PPARγ",
"search_terms": [
"Peroxisome proliferator-activated receptor gamma"
],
"group": "Nutritional sensors"
},

{
"uniprot_id": "O75469",
"nr_code": "NR1I2",
"common_name": "PXR",
"search_terms": [
"Nuclear receptor subfamily 1 group I member 2"
],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P20393",
"nr_code": "NR1D1",
"common_name": "Rev-erbα",
"search_terms": [
"Nuclear receptor subfamily 1 group D member 1"
],
"group": "Nutritional sensors"
},

{
"uniprot_id": "Q14995",
"nr_code": "NR1D2",
"common_name": "Rev-erbβ",
"search_terms": [
"Nuclear receptor subfamily 1 group D member 2"
],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P35398",
"nr_code": "NR1F1",
"common_name": "RORα",
"search_terms": ["Nuclear receptor ROR-alpha"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "Q92753",
"nr_code": "NR1F2",
"common_name": "RORβ",
"search_terms": ["Nuclear receptor ROR-beta"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P51449",
"nr_code": "NR1F3",
"common_name": "RORγ",
"search_terms": ["Nuclear receptor ROR-gamma"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P19793",
"nr_code": "NR2B1",
"common_name": "RXRα",
"search_terms": ["RETINOIC ACID RECEPTOR RXR-ALPHA"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P28702",
"nr_code": "NR2B2",
"common_name": "RXRβ",
"search_terms": ["Retinoic acid receptor RXR-beta"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P48443",
"nr_code": "NR2B3",
"common_name": "RXRγ",
"search_terms": ["Retinoic acid receptor RXR-gamma"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "Q13285",
"nr_code": "NR5A1",
"common_name": "SF-1",
"search_terms": ["Steroidogenic factor 1"],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P13056",
"nr_code": "NR2C1",
"common_name": "TR2",
"search_terms": [
"Nuclear receptor subfamily 2 group C member 1"
],
"group": "Nutritional sensors"
},

{
"uniprot_id": "P49116",
"nr_code": "NR2C2",
"common_name": "TR4",
"search_terms": [
"Nuclear receptor subfamily 2 group C member 2"
],
"group": "Nutritional sensors"
},


# =========================================================================
# ORPHAN RECEPTORS
# =========================================================================

{
"uniprot_id": "P10589",
"nr_code": "NR2F1",
"common_name": "COUP-TFI",
"search_terms": ["COUP transcription factor 1"],
"group": "Orphan receptors"
},

{
"uniprot_id": "P24468",
"nr_code": "NR2F2",
"common_name": "COUP-TFII",
"search_terms": ["COUP transcription factor 2"],
"group": "Orphan receptors"
},

{
"uniprot_id": "P51843",
"nr_code": "NR0B1",
"common_name": "DAX-1",
"search_terms": [
"Nuclear receptor subfamily 0 group B member 1"
],
"group": "Orphan receptors"
},

{
"uniprot_id": "P10588",
"nr_code": "NR2F6",
"common_name": "EAR-2",
"search_terms": [
"Nuclear receptor subfamily 2 group F member 6"
],
"group": "Orphan receptors"
},

{
"uniprot_id": "Q15406",
"nr_code": "NR6A1",
"common_name": "GCNF",
"search_terms": [
"Nuclear receptor subfamily 6 group A member 1"
],
"group": "Orphan receptors"
},

{
"uniprot_id": "Q92570",
"nr_code": "NR4A3",
"common_name": "NOR-1",
"search_terms": [
"Nuclear receptor subfamily 4 group A member 3"
],
"group": "Orphan receptors"
},

{
"uniprot_id": "Q9Y5X4",
"nr_code": "NR2E3",
"common_name": "PNR",
"search_terms": ["Photoreceptor-specific nuclear receptor"],
"group": "Orphan receptors"
},

{
"uniprot_id": "Q15466",
"nr_code": "NR0B2",
"common_name": "SHP",
"search_terms": [
"Nuclear receptor subfamily 0 group B member 2"
],
"group": "Orphan receptors"
},

{
"uniprot_id": "Q9Y466",
"nr_code": "NR2E1",
"common_name": "TLX",
"search_terms": [
"Nuclear receptor subfamily 2 group E member 1"
],
"group": "Orphan receptors"
}
]

In [ ]:
meta=pd.DataFrame(NR_METADATA)
assert len(meta)==48,f'Expected 48 nuclear receptors, found {len(meta)}'
assert meta['uniprot_id'].is_unique

UID_TO_NAME=dict(zip(meta['uniprot_id'],meta['common_name']))
UID_TO_CODE=dict(zip(meta['uniprot_id'],meta['nr_code']))
UID_TO_GROUP=dict(zip(meta['uniprot_id'],meta['group']))

print('NR metadata:',len(meta),'receptors')

# Part I — Student A handoff audit

Student B geometry must not start until the input receptor-chain assignments and canonical numbering are valid.

The critical failure mode from the previous cohort was GR: crystallographic constructs can use local construct numbering, while biological comparison requires full-length UniProt positions. Therefore, every retained receptor chain is sequence-aligned to the canonical UniProt sequence before any geometry is computed.

## 2. Load the supplied snapshot

In [ ]:
required=[
    'all_nr_structures.csv',
    'basic_filtered_structures.csv',
    'confirmed_cofactor_complexes_FIXED.csv'
]
missing=[x for x in required if not (SNAPSHOT/x).exists()]
assert not missing,f'Missing source files: {missing}'

allnr=pd.read_csv(SNAPSHOT/'all_nr_structures.csv')
basic=pd.read_csv(SNAPSHOT/'basic_filtered_structures.csv')
conf0=pd.read_csv(SNAPSHOT/'confirmed_cofactor_complexes_FIXED.csv')

for df in [allnr,basic,conf0]:
    df['pdb_id']=df['pdb_id'].astype(str).str.upper().str.strip()

print('Starting inventory unique PDBs:',allnr['pdb_id'].nunique())
print('Basic-filter unique PDBs:',basic['pdb_id'].nunique())
print('Confirmed cofactor unique PDBs:',conf0['pdb_id'].nunique())

## 3. All-48 RCSB update audit

Re-query RCSB using all 48 UniProt IDs. New PDB IDs are **not automatically admitted**. They must pass the same domain/cofactor criteria as the original curated dataset.

The completed audit found five additional TR4-related structures. They were reviewed explicitly: four do not enter the primary cofactor geometry dataset, and the JAZF1 complex is retained only as a special noncanonical case rather than mixed into the primary LXXLL/CoRNR analysis.

In [ ]:
RCSB_SEARCH='https://search.rcsb.org/rcsbsearch/v2/query'

def query_rcsb_uniprot(uid):
    payload={
        'query':{
            'type':'terminal',
            'service':'text',
            'parameters':{
                'attribute':'rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession',
                'operator':'exact_match',
                'value':uid
            }
        },
        'return_type':'entry',
        'request_options':{'paginate':{'start':0,'rows':10000}}
    }
    try:
        r=requests.post(RCSB_SEARCH,json=payload,timeout=90)
        r.raise_for_status()
        ids=sorted({x['identifier'] for x in r.json().get('result_set',[])})
        return uid,ids,'PASS',''
    except Exception as e:
        return uid,[],'ERROR',str(e)

requery_rows=[]
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futs={ex.submit(query_rcsb_uniprot,u):u for u in meta['uniprot_id']}
    for fut in tqdm(as_completed(futs),total=len(futs),desc='RCSB re-query'):
        uid,ids,status,error=fut.result()
        name=UID_TO_NAME[uid]
        if ids:
            requery_rows.extend({
                'uniprot_id':uid,'receptor':name,'pdb_id':p,'status':status,'error':error
            } for p in ids)
        else:
            requery_rows.append({
                'uniprot_id':uid,'receptor':name,'pdb_id':np.nan,'status':status,'error':error
            })

requery=pd.DataFrame(requery_rows)
requery.to_csv(PROCESSED/'studentA_all48_pdb_requery.csv',index=False)

old_pdbs=set(allnr['pdb_id'].dropna())
queried_new=sorted(set(requery['pdb_id'].dropna())-old_pdbs)
pd.DataFrame({'pdb_id':queried_new}).to_csv(
    PROCESSED/'studentA_new_pdbs_to_review.csv',index=False
)

print('RCSB query errors:',(requery['status']=='ERROR').sum())
print('New PDB IDs requiring review:',len(queried_new))

In [ ]:
NEW_PDB_REVIEW=pd.DataFrame([
    ['3P0U','TR4','P49116','LBD',False,'','EXCLUDE_NO_COFACTOR',
     'Ligand-free/autorepressed TR4 LBD; no separate cofactor peptide.'],
    ['7XV6','TR4','P49116','DBD',False,'','EXCLUDE_DBD',
     'TR4 DNA-binding-domain structure.'],
    ['7XV8','TR4','P49116','DBD',False,'','EXCLUDE_DBD',
     'TR4 DNA-binding-domain homodimer bound to DNA.'],
    ['7XV9','TR4','P49116','DBD',False,'','EXCLUDE_DBD',
     'TR4 DNA-binding-domain structure.'],
    ['7XVA','TR4','P49116','LBD',True,'JAZF1','EXCLUDE_PRIMARY_KEEP_SPECIAL',
     'TR4 LBD + JAZF1 corepressor fragment; noncanonical interface, retained separately.']
],columns=[
    'pdb_id','receptor','uniprot_id','domain','cofactor_present','cofactor',
    'decision','reason'
])
NEW_PDB_REVIEW.to_csv(PROCESSED/'studentA_new_pdb_review.csv',index=False)

unreviewed=sorted(set(queried_new)-set(NEW_PDB_REVIEW['pdb_id']))
assert not unreviewed,f'Unreviewed new PDBs remain: {unreviewed}'
print('New PDB review complete:',len(queried_new),'reviewed')

## 4. Assign each cofactor peptide to its physical receptor chain

In [ ]:
def contact_counts(s):
    c=Counter()
    if pd.isna(s):
        return c
    for x in str(s).split(';'):
        x=x.strip()
        if ':' in x:
            c[x.split(':',1)[0].strip()]+=1
    return c

def pick_chain(s):
    c=contact_counts(s)
    if not c:
        return np.nan
    m=max(c.values())
    top=[ch for ch,n in c.items() if n==m]
    return top[0] if len(top)==1 else np.nan

conf0=conf0.copy()
conf0['receptor_chain']=conf0['contact_residues'].apply(pick_chain)

print('missing receptor-chain assignments:',conf0['receptor_chain'].isna().sum())
assert conf0['receptor_chain'].notna().all()

## 5. Download/resume the confirmed mmCIF structures

In [ ]:
final_pdbs=sorted(conf0['pdb_id'].dropna().unique())
print('unique confirmed PDBs:',len(final_pdbs))

def download_cif(pdb):
    out=STRUCTURES/f'{pdb}.cif'
    if out.exists() and out.stat().st_size>1000:
        return {'pdb_id':pdb,'status':'exists','error':''}

    url=f'https://files.rcsb.org/download/{pdb}.cif'
    last=''
    for attempt in range(3):
        try:
            r=requests.get(url,timeout=60)
            r.raise_for_status()
            out.write_bytes(r.content)
            return {'pdb_id':pdb,'status':'downloaded','error':''}
        except Exception as e:
            last=str(e)
            time.sleep(2)
    return {'pdb_id':pdb,'status':'ERROR','error':last}

download_rows=[]
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futs={ex.submit(download_cif,p):p for p in final_pdbs}
    for fut in tqdm(as_completed(futs),total=len(futs),desc='mmCIF files'):
        download_rows.append(fut.result())

download_log=pd.DataFrame(download_rows).sort_values('pdb_id')
download_log.to_csv(PROCESSED/'structure_download_log.csv',index=False)

print(download_log['status'].value_counts())
assert not (download_log['status']=='ERROR').any()

## 6. Download canonical UniProt sequences for all 48 receptors

All 48 canonical sequences are cached in JSON. The geometry subset may contain fewer receptor types, but the registry itself is complete.

In [ ]:
all_uids=sorted(meta['uniprot_id'].unique())
seq_file=NUMBERING/'uniprot_sequences_all48.json'

if seq_file.exists():
    with open(seq_file,encoding='utf-8') as f:
        seqs=json.load(f)
else:
    seqs={}

def fetch_uniprot(uid):
    if uid in seqs and seqs[uid]:
        return uid,seqs[uid],'exists',''
    try:
        url=f'https://rest.uniprot.org/uniprotkb/{uid}.fasta'
        r=requests.get(url,timeout=60)
        r.raise_for_status()
        lines=r.text.strip().splitlines()
        return uid,''.join(lines[1:]),'downloaded',''
    except Exception as e:
        return uid,'','ERROR',str(e)

seq_rows=[]
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futs={ex.submit(fetch_uniprot,u):u for u in all_uids}
    for fut in tqdm(as_completed(futs),total=len(futs),desc='UniProt sequences (all 48)'):
        uid,seq,status,error=fut.result()
        if seq:
            seqs[uid]=seq
        seq_rows.append({
            'uniprot_id':uid,'receptor':UID_TO_NAME[uid],
            'length':len(seq),'status':status,'error':error
        })

with open(seq_file,'w',encoding='utf-8') as f:
    json.dump(seqs,f,indent=2)

# keep the historical filename too, because later cells and group handoffs may expect it
with open(NUMBERING/'uniprot_sequences.json','w',encoding='utf-8') as f:
    json.dump(seqs,f,indent=2)

seq_qc=pd.DataFrame(seq_rows)
seq_qc.to_csv(PROCESSED/'studentA_uniprot_all48_qc.csv',index=False)

missing_seq=sorted(set(all_uids)-set(k for k,v in seqs.items() if v))
print('canonical sequences available:',len(set(all_uids)-set(missing_seq)),'/ 48')
print('missing:',missing_seq)
assert not missing_seq,'All 48 canonical sequences are required before final pre-geometry sign-off'

## 7. Sequence-identify receptor chains and create canonical numbering maps

For every candidate long receptor chain:

1. Extract resolved amino-acid residues from the mmCIF structure.
2. Align the resolved sequence to candidate canonical UniProt sequences.
3. Select the best receptor identity using coverage × identity.
4. Save the mapping `PDB residue label → canonical UniProt position`.
5. Flag low-confidence assignments for curation instead of silently using them.

In [ ]:
CUSTOM_AA={'MSE':'M','SEP':'S','TPO':'T','PTR':'Y'}

def make_aligner():
    a=PairwiseAligner()
    a.mode='local'
    a.match_score=2
    a.mismatch_score=-1
    a.open_gap_score=-10
    a.extend_gap_score=-0.5
    return a

def get_chain_residues(st,ch):
    chain=st[0][ch]
    out=[]
    for r in chain:
        if not is_aa(r,standard=False) or 'CA' not in r:
            continue
        aa=seq1(r.get_resname(),custom_map=CUSTOM_AA,undef_code='X')
        out.append((r,aa))
    return out

def align_chain(st,ch,canon,aligner):
    residues=get_chain_residues(st,ch)
    seq=''.join(x[1] for x in residues)
    if not seq:
        return None

    aln=aligner.align(canon,seq)[0]
    pairs={}
    for cb,pb in zip(aln.aligned[0],aln.aligned[1]):
        cs,ce=cb
        ps,pe=pb
        for ci,pi in zip(range(cs,ce),range(ps,pe)):
            pairs[pi]=ci

    mp={}
    matches=0
    for pi,ci in pairs.items():
        r=residues[pi][0]
        label=f"{r.id[1]}{r.id[2].strip()}"
        mp[label]=ci+1
        if seq[pi]==canon[ci]:
            matches+=1

    n=len(residues)
    nm=len(pairs)
    coverage=nm/n if n else 0
    identity=matches/nm if nm else 0
    return {
        'map':mp,
        'coverage':coverage,
        'identity':identity,
        'metric':coverage*identity,
        'n_resolved':n,
        'n_mapped':nm,
        'score':float(aln.score)
    }

def parse_chain_lengths(s):
    out={}
    if pd.isna(s):
        return out
    for x in str(s).split(';'):
        if ':' not in x:
            continue
        ch,n=x.split(':',1)
        try:
            out[ch.strip()]=int(n)
        except ValueError:
            pass
    return out

basic_final=basic[basic['pdb_id'].isin(final_pdbs)].copy()
pdb_chains=(
    basic_final[['pdb_id','receptor_chains']]
    .drop_duplicates('pdb_id')
    .copy()
)
pdb_chains['chain_dict']=pdb_chains['receptor_chains'].apply(parse_chain_lengths)

In [ ]:
def parse_chain_lengths(s):
    out={}
    if pd.isna(s):
        return out
    for part in str(s).split(';'):
        part=part.strip()
        if ':' not in part:
            continue
        ch,n=part.split(':',1)
        try:
            out[ch.strip()]=int(float(n))
        except Exception:
            continue
    return out

final_pdbs=set(conf0['pdb_id'].dropna().astype(str).str.upper())
basic_final=basic[basic['pdb_id'].astype(str).str.upper().isin(final_pdbs)].copy()

pdb_chain_rows=[]
for r in basic_final.itertuples(index=False):
    d=parse_chain_lengths(getattr(r,'receptor_chains',np.nan))
    if d:
        pdb_chain_rows.append({'pdb_id':r.pdb_id,'chain_dict':d})

pdb_chains=pd.DataFrame(pdb_chain_rows).drop_duplicates('pdb_id')
print('PDBs with candidate receptor chains:',len(pdb_chains))

In [ ]:
def identify_pdb_chains(row):
    pdb=row['pdb_id']
    chain_dict=row['chain_dict']
    out=[]
    maps={}
    try:
        parser=MMCIFParser(QUIET=True,auth_chains=True,auth_residues=True)
        st=parser.get_structure(pdb,STRUCTURES/f'{pdb}.cif')
        aligner=make_aligner()

        candidates=(
            nr.loc[nr['pdb_id']==pdb,['receptor','uniprot_id']]
              .drop_duplicates()
        )

        for ch,n_chain in chain_dict.items():
            scores=[]
            for c in candidates.itertuples(index=False):
                if c.uniprot_id not in seqs:
                    continue
                try:
                    a=align_chain(st,ch,seqs[c.uniprot_id],aligner)
                except Exception:
                    continue
                if a is not None:
                    scores.append({'receptor':c.receptor,'uniprot_id':c.uniprot_id,**a})

            if not scores:
                out.append({
                    'pdb_id':pdb,'receptor_chain':ch,'chain_length':n_chain,
                    'status':'ERROR','error':'no sequence match'
                })
                continue

            scores.sort(key=lambda x:x['metric'],reverse=True)
            best=scores[0]
            second=scores[1]['metric'] if len(scores)>1 else np.nan
            margin=best['metric']-second if len(scores)>1 else np.nan
            status='PASS' if best['coverage']>=0.80 and best['identity']>=0.90 else 'REVIEW'

            out.append({
                'pdb_id':pdb,'receptor_chain':ch,'chain_length':n_chain,
                'receptor':best['receptor'],'uniprot_id':best['uniprot_id'],
                'coverage':best['coverage'],'identity':best['identity'],
                'metric':best['metric'],'second_metric':second,'margin':margin,
                'n_resolved':best['n_resolved'],'n_mapped':best['n_mapped'],
                'status':status,'error':''
            })
            maps[f"{pdb}|{best['receptor']}|{ch}"]=best['map']

        return out,maps,''
    except Exception as e:
        return [],{},str(e)

chain_rows=[]
numbering_maps={}
chain_errors=[]

records=pdb_chains.to_dict('records')
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futs={ex.submit(identify_pdb_chains,r):r['pdb_id'] for r in records}
    for fut in tqdm(as_completed(futs),total=len(futs),desc='Receptor chain identity'):
        pdb=futs[fut]
        try:
            rows,maps,error=fut.result()
            chain_rows.extend(rows)
            numbering_maps.update(maps)
            if error:
                chain_errors.append({'pdb_id':pdb,'error':error})
        except Exception as e:
            chain_errors.append({'pdb_id':pdb,'error':str(e),'traceback':traceback.format_exc()})

chain_id=pd.DataFrame(chain_rows)
print(chain_id['status'].value_counts(dropna=False))

chain_id.to_csv(PROCESSED/'receptor_chain_identity.csv',index=False)
pd.DataFrame(chain_errors).to_csv(PROCESSED/'receptor_chain_identity_errors.csv',index=False)

with open(NUMBERING/'canonical_numbering_maps_v2.json','w',encoding='utf-8') as f:
    json.dump(numbering_maps,f,indent=2)

### Manual receptor-chain curation decisions

The broad chain scan can contain candidate chains that are not actually used by the cofactor handoff. Only receptor chains used by the physical peptide–receptor pairs must be resolved before geometry.

Documented decisions:
- keep engineered ESR1 variants 7NEL/7NFB;
- exclude false SHP-as-receptor assignments;
- exclude non-human/wrong receptor chains in 7AOS and 1XLS.

In [ ]:
conf=conf0.copy()
conf=conf.rename(columns={'receptor':'source_receptor'})

src=(
    conf.groupby(['pdb_id','peptide_chain','receptor_chain'])['source_receptor']
        .agg(lambda x:';'.join(sorted(set(map(str,x)))))
        .reset_index(name='source_receptor_labels')
)

conf=(
    conf.drop_duplicates(['pdb_id','peptide_chain','receptor_chain'])
        .drop(columns=['source_receptor'],errors='ignore')
        .merge(src,on=['pdb_id','peptide_chain','receptor_chain'],how='left')
        .merge(
            chain_id[[
                'pdb_id','receptor_chain','receptor','uniprot_id',
                'coverage','identity','status'
            ]],
            on=['pdb_id','receptor_chain'],how='left'
        )
)

assert conf['receptor'].notna().all()

conf['curation_status']='KEEP'
conf['curation_note']=''

# Human ESR1 engineered variants: retain.
m=conf['pdb_id'].isin(['7NEL','7NFB'])
conf.loc[m,'curation_status']='KEEP_ENGINEERED_MUTANT'
conf.loc[m,'curation_note']='Human ESR1 engineered variant; full sequence coverage but reduced identity due to mutations'

# SHP was a coregulator / source-label artefact, not the long receptor chain.
shp_false=['2Q3Y','5UFS','6W9M','7YXC','7YXD','7YXN','7YXO','7YXP','7YXR','9HDF']
m=conf['pdb_id'].isin(shp_false) & (conf['receptor']=='SHP')
conf.loc[m,'curation_status']='EXCLUDE_FALSE_RECEPTOR'
conf.loc[m,'curation_note']='SHP assignment is incorrect: SHP is a coregulator peptide or the long chain is a glucocorticoid/ancestral corticoid receptor'

# Non-human receptor chains identified during manual review.
m=(conf['pdb_id']=='7AOS') & (conf['receptor_chain']=='A')
conf.loc[m,'curation_status']='EXCLUDE_NONHUMAN_RECEPTOR'
conf.loc[m,'curation_note']='Chain A is not the human RARA receptor chain'

m=(conf['pdb_id']=='1XLS') & conf['receptor_chain'].isin(['E','F','G','H'])
conf.loc[m,'curation_status']='EXCLUDE_NONHUMAN_RECEPTOR'
conf.loc[m,'curation_note']='Chains E-H are not human RXRA receptor chains'

unresolved=conf[(conf['status']=='REVIEW') & (conf['curation_status']=='KEEP')]
assert unresolved.empty,f'Unresolved REVIEW chains remain: {len(unresolved)}'

conf_clean=conf[conf['curation_status'].str.startswith('KEEP')].copy()
conf_clean.to_csv(PROCESSED/'cofactor_complexes_handoff_clean.csv',index=False)

print('physical peptide rows before curation:',len(conf))
print('physical peptide rows after curation:',len(conf_clean))
print(conf['curation_status'].value_counts())

In [ ]:
def join_unique(s):
    vals={str(x).strip() for x in s.dropna() if str(x).strip()}
    return ';'.join(sorted(vals))

geom=(
    conf_clean.groupby(
        ['pdb_id','receptor','uniprot_id','receptor_chain'],
        as_index=False
    )
    .agg(
        peptide_chains=('peptide_chain',join_unique),
        n_peptides=('peptide_chain','nunique'),
        cofactor_types=('cofactor_type',join_unique),
        cofactor_identities=('cofactor_identity',join_unique)
    )
)

geom.to_csv(PROCESSED/'geometry_structures_handoff_clean.csv',index=False)

print('geometry units:',len(geom))
print('unique PDBs:',geom['pdb_id'].nunique())
print('receptors:',geom['receptor'].nunique())

## 8. Pre-geometry canonical-numbering gate

This is the hard boundary between Student A infrastructure and Student B geometry.

Every geometry unit must:
- have ≥80% sequence-mapping coverage;
- have a saved canonical numbering map;
- pass the GR full-length numbering sanity check.

If any retained row fails, the notebook stops.

In [ ]:
with open(NUMBERING/'canonical_numbering_maps_v2.json',encoding='utf-8') as f:
    numbering_maps=json.load(f)

pregeom=geom.merge(
    chain_id[[
        'pdb_id','receptor_chain','receptor','uniprot_id',
        'coverage','identity','status'
    ]],
    on=['pdb_id','receptor_chain','receptor','uniprot_id'],
    how='left'
)

def map_key(r):
    return f"{r['pdb_id']}|{r['receptor']}|{r['receptor_chain']}"

pregeom['numbering_map_saved']=pregeom.apply(
    lambda r:map_key(r) in numbering_maps,axis=1
)

pregeom['canonical_ok']=(
    pregeom['coverage'].fillna(0).ge(0.80)
    & pregeom['numbering_map_saved']
)

# GR-specific full-length sanity check
def gr_map_ok(r):
    if str(r['uniprot_id'])!='P04150':
        return True
    mp=numbering_maps.get(map_key(r),{})
    vals=[int(v) for v in mp.values()] if mp else []
    return bool(vals) and max(vals)>=700 and min(vals)>=400

pregeom['gr_full_length_numbering_ok']=pregeom.apply(gr_map_ok,axis=1)
pregeom['pregeometry_status']=np.where(
    pregeom['canonical_ok'] & pregeom['gr_full_length_numbering_ok'],
    'PASS','FAIL'
)

pregeom.to_csv(PROCESSED/'pregeometry_validation_report.csv',index=False)

print(pregeom['pregeometry_status'].value_counts())
print('GR rows:',
      (pregeom['uniprot_id']=='P04150').sum())
print('GR numbering failures:',
      ((pregeom['uniprot_id']=='P04150') & ~pregeom['gr_full_length_numbering_ok']).sum())

assert (pregeom['pregeometry_status']=='PASS').all(),     'STOP: Student B geometry cannot start until every geometry row passes canonical-numbering validation'

print('\nGEOMETRY DATASET LOCKED:')
print('geometry units:',len(geom))
print('unique PDBs:',geom['pdb_id'].nunique())
print('receptors:',geom['uniprot_id'].nunique())

# Part II — Student B geometry infrastructure

## 9. Run/resume P2Rank

In [ ]:
assert PRANK.exists(),(
    'P2Rank executable not found. Set environment variable P2RANK_HOME '
    'to the extracted P2Rank 2.4.2 directory.'
)

In [ ]:
def run_p2rank(pdb):
    cif=STRUCTURES/f'{pdb}.cif'
    out=P2OUT/pdb
    pred=out/f'{pdb}.cif_predictions.csv'

    if pred.exists():
        return {'pdb_id':pdb,'status':'exists','runtime_s':0.0,'error':''}
    if not cif.exists():
        return {'pdb_id':pdb,'status':'ERROR','runtime_s':0.0,'error':'CIF missing'}

    out.mkdir(parents=True,exist_ok=True)
    cmd=f'"{PRANK}" predict -f "{cif}" -o "{out}"'
    t=time.time()

    try:
        r=subprocess.run(
            cmd,shell=True,capture_output=True,text=True,
            encoding='utf-8',errors='replace',timeout=600
        )
        sec=time.time()-t
        (out/'run_stdout.txt').write_text(r.stdout,encoding='utf-8')
        (out/'run_stderr.txt').write_text(r.stderr,encoding='utf-8')

        if r.returncode==0 and pred.exists():
            return {'pdb_id':pdb,'status':'success','runtime_s':sec,'error':''}
        return {
            'pdb_id':pdb,'status':'ERROR','runtime_s':sec,
            'error':r.stderr[-500:].replace('\n',' ')
        }
    except Exception as e:
        return {'pdb_id':pdb,'status':'ERROR','runtime_s':time.time()-t,'error':str(e)}

pdbs=sorted(geom['pdb_id'].unique())
p2_rows=[]

with ThreadPoolExecutor(max_workers=P2RANK_WORKERS) as ex:
    futs={ex.submit(run_p2rank,p):p for p in pdbs}
    for fut in tqdm(as_completed(futs),total=len(futs),desc='P2Rank'):
        p2_rows.append(fut.result())

p2log=pd.DataFrame(p2_rows).sort_values('pdb_id')
p2log.to_csv(PROCESSED/'p2rank_batch_log.csv',index=False)

print(p2log['status'].value_counts())
assert not (p2log['status']=='ERROR').any()

## 10. Select the receptor-chain-dominant pocket and map it to canonical numbering

P2Rank ranks pockets globally across a structure. In multichain structures, the top global pocket can belong mainly to a neighbouring chain. Therefore, the pipeline:

1. parses every P2Rank pocket;
2. keeps only pockets containing the target receptor chain;
3. selects the highest-ranked pocket with ≥80% of pocket residues on the target receptor chain;
4. maps those deposited residue labels to canonical UniProt positions.

The 80% chain fraction is a conservative QC heuristic introduced after manual validation of difficult multichain cases.

In [ ]:
with open(NUMBERING/'canonical_numbering_maps_v2.json',encoding='utf-8') as f:
    numbering_maps=json.load(f)

def parse_residue_ids(s):
    if pd.isna(s):
        return []
    out=[]
    for x in str(s).split():
        if '_' not in x:
            continue
        ch,res=x.split('_',1)
        out.append({'chain':ch,'pdb_residue':res})
    return out

def get_chain_pockets(pred,chain):
    pred=pred.copy()
    pred.columns=pred.columns.str.strip()
    out=[]
    for _,r in pred.iterrows():
        residues=parse_residue_ids(r['residue_ids'])
        chain_res=[x['pdb_residue'] for x in residues if x['chain']==chain]
        if not chain_res:
            continue
        out.append({
            'rank':int(r['rank']),
            'score':float(r['score']),
            'probability':float(r['probability']),
            'center_x':float(r['center_x']),
            'center_y':float(r['center_y']),
            'center_z':float(r['center_z']),
            'pdb_residues':chain_res,
            'n_chain_residues':len(chain_res),
            'n_total_residues':len(residues),
            'chain_fraction':len(chain_res)/len(residues)
        })
    return sorted(out,key=lambda x:x['rank'])

def select_chain_pocket(pred,chain,min_fraction=0.80):
    candidates=get_chain_pockets(pred,chain)
    if not candidates:
        return None,'NO_CHAIN_POCKET'
    good=[x for x in candidates if x['chain_fraction']>=min_fraction]
    return (good[0],'PASS') if good else (candidates[0],'REVIEW_NO_CHAIN_DOMINANT_POCKET')

def canonicalize_pocket(pdb,receptor,chain,pdb_residues):
    key=f'{pdb}|{receptor}|{chain}'
    mp=numbering_maps.get(key,{})
    canon=[]
    missing=[]
    for res in map(str,pdb_residues):
        if res in mp:
            canon.append(mp[res])
        else:
            missing.append(res)
    return canon,missing

In [ ]:
def process_pocket_row(row):
    pdb=row['pdb_id']
    rec=row['receptor']
    ch=row['receptor_chain']
    base={
        'pdb_id':pdb,'receptor':rec,
        'uniprot_id':row['uniprot_id'],'receptor_chain':ch
    }
    try:
        pred=pd.read_csv(P2OUT/pdb/f'{pdb}.cif_predictions.csv')
        pred.columns=pred.columns.str.strip()
        pocket,status=select_chain_pocket(pred,ch,min_fraction=0.80)

        if pocket is None:
            return {**base,'status':status,'error':''}

        canon,missing=canonicalize_pocket(pdb,rec,ch,pocket['pdb_residues'])
        return {
            **base,
            'selected_pocket_rank':pocket['rank'],
            'p2rank_score':pocket['score'],
            'p2rank_probability':pocket['probability'],
            'p2rank_center_x':pocket['center_x'],
            'p2rank_center_y':pocket['center_y'],
            'p2rank_center_z':pocket['center_z'],
            'n_chain_residues':pocket['n_chain_residues'],
            'n_total_pocket_residues':pocket['n_total_residues'],
            'chain_fraction':pocket['chain_fraction'],
            'pocket_residues_pdb':';'.join(map(str,pocket['pdb_residues'])),
            'pocket_residues_canonical':';'.join(map(str,canon)),
            'n_canonical_residues':len(canon),
            'n_unmapped_residues':len(missing),
            'unmapped_residues':';'.join(map(str,missing)),
            'status':status,'error':''
        }
    except Exception as e:
        return {**base,'status':'ERROR','error':str(e)}

items=geom.to_dict('records')
pocket_rows=[]

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futs={ex.submit(process_pocket_row,r):(r['pdb_id'],r['receptor_chain']) for r in items}
    for fut in tqdm(as_completed(futs),total=len(futs),desc='Pocket parsing'):
        try:
            pocket_rows.append(fut.result())
        except Exception as e:
            pdb,ch=futs[fut]
            pocket_rows.append({
                'pdb_id':pdb,'receptor_chain':ch,
                'status':'ERROR','error':str(e)
            })

pockets=pd.DataFrame(pocket_rows)
pockets['mapping_fraction']=np.where(
    pockets['n_chain_residues'].fillna(0)>0,
    pockets['n_canonical_residues']/pockets['n_chain_residues'],
    np.nan
)

print(pockets['status'].value_counts(dropna=False))
print('chain fraction <0.8:',(pockets['chain_fraction']<0.8).sum())
print('nothing mapped:',(pockets['n_canonical_residues']==0).sum())
print('mapping <80%:',(pockets['mapping_fraction']<0.8).sum())

pockets.to_csv(PROCESSED/'p2rank_pockets_canonical_final.csv',index=False)
pockets.to_parquet(PROCESSED/'p2rank_pockets_canonical_final.parquet',index=False)

In [ ]:
uid_name=UID_TO_NAME
uid_code=UID_TO_CODE
uid_group=UID_TO_GROUP

missing_meta=sorted(set(pockets['uniprot_id'].dropna())-set(meta['uniprot_id']))
assert not missing_meta,f'Missing NR_METADATA entries: {missing_meta}'

pockets=pockets.rename(columns={'receptor':'receptor_original'})
pockets['receptor']=pockets['uniprot_id'].map(uid_name)
pockets['nr_code']=pockets['uniprot_id'].map(uid_code)
pockets['nr_group']=pockets['uniprot_id'].map(uid_group)

meta.to_csv(PROCESSED/'nr_metadata.csv',index=False)
pockets.to_csv(PROCESSED/'p2rank_pockets_final_named.csv',index=False)
pockets.to_parquet(PROCESSED/'p2rank_pockets_final_named.parquet',index=False)

print('final pocket rows:',len(pockets))
print('receptors:',pockets['uniprot_id'].nunique())

### Pocket QC gate

Expected completed-run checks:
- 1,167 pocket rows;
- 0 chain-fraction failures below 0.80;
- 0 pockets with no canonical residues;
- 0 rows below 80% canonical mapping.

In [ ]:
assert (pockets['status']=='PASS').all()
assert not (pockets['chain_fraction']<0.80).any()
assert not (pockets['n_canonical_residues']==0).any()
assert not (pockets['mapping_fraction']<0.80).any()
print('Pocket QC PASS:',len(pockets),'/',len(pockets))

## 11. SASA-weighted pocket centroid

The centroid function takes **canonical pocket positions as its scientific input**. Those positions are mapped back to deposited residue coordinates only to retrieve 3D coordinates.

For each pocket residue:
- use Cβ as the reference atom (Cα for glycine);
- compute relative exposure with BioPython Shrake–Rupley;
- weight more buried residues more strongly;
- calculate the weighted mean 3D coordinate.

The output also records how far this centroid is from the P2Rank-reported center for QC.

> **Method note:** the current implementation uses Shrake–Rupley on the receptor chain. If the supervisor requires DSSP/full-complex SASA specifically, change the SASA backend before manuscript-final analysis; do not silently mix methods.

In [ ]:
MAX_ASA={
    'ALA':129,'ARG':274,'ASN':195,'ASP':193,'CYS':167,
    'GLN':225,'GLU':223,'GLY':104,'HIS':224,'ILE':197,
    'LEU':201,'LYS':236,'MET':224,'PHE':240,'PRO':159,
    'SER':155,'THR':172,'TRP':285,'TYR':263,'VAL':174
}

def residue_label(r):
    return f"{r.id[1]}{r.id[2].strip()}"

def parse_semicolon_list(s):
    if pd.isna(s) or not str(s).strip():
        return []
    return [x.strip() for x in str(s).split(';') if x.strip()]

def canonical_to_pdb_map(row):
    rec=row.get('receptor_original',row.get('receptor'))
    key=f"{row['pdb_id']}|{rec}|{row['receptor_chain']}"
    mp=numbering_maps.get(key,{})
    return {str(v):str(k) for k,v in mp.items()}

def centroid_for_chain(st,row):
    pdb=row['pdb_id']
    chain_id=row['receptor_chain']
    base={
        'pdb_id':pdb,'uniprot_id':row['uniprot_id'],
        'receptor':row['receptor'],'nr_code':row.get('nr_code'),
        'nr_group':row.get('nr_group'),'receptor_chain':chain_id
    }

    try:
        if chain_id not in st[0]:
            raise ValueError('receptor chain missing')

        chain=st[0][chain_id]

        # CANONICAL POSITIONS ARE THE SCIENTIFIC INPUT
        canonical_positions=set(parse_semicolon_list(
            row['pocket_residues_canonical']
        ))
        c2p=canonical_to_pdb_map(row)

        wanted_pdb={c2p[c] for c in canonical_positions if c in c2p}
        if not wanted_pdb:
            raise ValueError('canonical pocket residues could not be mapped back to PDB coordinates')

        sr=ShrakeRupley()
        sr.compute(chain,level='R')

        coords=[]
        weights=[]
        used_canonical=[]
        skipped=[]

        p2c={v:k for k,v in c2p.items()}

        for r in chain:
            if r.id[0]!=' ':
                continue
            lab=residue_label(r)
            if lab not in wanted_pdb:
                continue

            canon=p2c.get(lab)
            rn=r.get_resname().upper()

            if rn not in MAX_ASA:
                skipped.append(f'{canon}:{lab}:nonstandard')
                continue

            sasa=getattr(r,'sasa',None)
            if sasa is None:
                skipped.append(f'{canon}:{lab}:no_sasa')
                continue

            rsa=np.clip(float(sasa)/MAX_ASA[rn],0.0,1.0)
            w=max(1.0-rsa,0.0)
            atom='CA' if rn=='GLY' else 'CB'

            if atom not in r:
                if 'CA' in r:
                    atom='CA'
                else:
                    skipped.append(f'{canon}:{lab}:no_reference_atom')
                    continue

            coords.append(r[atom].get_coord().astype(float))
            weights.append(w)
            used_canonical.append(str(canon))

        if not coords:
            raise ValueError('no usable pocket residues')

        coords=np.asarray(coords,float)
        weights=np.asarray(weights,float)

        if weights.sum()<=1e-12:
            weights=np.ones(len(weights),dtype=float)

        c=np.average(coords,axis=0,weights=weights)
        p2=np.array([
            row['p2rank_center_x'],
            row['p2rank_center_y'],
            row['p2rank_center_z']
        ],dtype=float)

        return {
            **base,
            'centroid_x':float(c[0]),
            'centroid_y':float(c[1]),
            'centroid_z':float(c[2]),
            'n_requested':len(canonical_positions),
            'n_used':len(used_canonical),
            'n_skipped':len(skipped),
            'used_canonical_residues':';'.join(used_canonical),
            'skipped_residues':';'.join(skipped),
            'centroid_shift_from_p2rank_A':float(np.linalg.norm(c-p2)),
            'status':'PASS','error':''
        }

    except Exception as e:
        return {
            **base,
            'centroid_x':np.nan,'centroid_y':np.nan,'centroid_z':np.nan,
            'n_requested':np.nan,'n_used':np.nan,'n_skipped':np.nan,
            'used_canonical_residues':'','skipped_residues':'',
            'centroid_shift_from_p2rank_A':np.nan,
            'status':'ERROR','error':str(e)
        }

def centroid_pdb_worker(pdb,rows):
    parser=MMCIFParser(QUIET=True,auth_chains=True,auth_residues=True)
    try:
        st=parser.get_structure(pdb,STRUCTURES/f'{pdb}.cif')
        return [centroid_for_chain(st,row) for row in rows]
    except Exception as e:
        return [{
            'pdb_id':pdb,'uniprot_id':row.get('uniprot_id'),
            'receptor':row.get('receptor'),'nr_code':row.get('nr_code'),
            'nr_group':row.get('nr_group'),
            'receptor_chain':row.get('receptor_chain'),
            'status':'ERROR','error':str(e)
        } for row in rows]

## 12. Centroid validation: up to five structures per receptor

The supervisor requires validation before scaling. A deterministic sample of up to five receptor-chain units per receptor is selected.

Validation has three layers:
1. a small manual PyMOL sanity-check set;
2. scripted PyMOL measurements for all validation cases;
3. manual review of only the automatically flagged outliers.

This replaces 121 repetitive visual checks with a reproducible measurement-based screen while retaining manual confirmation for unusual cases.

In [ ]:
CENTROID_VAL=VALIDATION/'centroid'
CENTROID_VAL.mkdir(parents=True,exist_ok=True)

val_candidates=(
    pockets.sort_values(
        ['uniprot_id','p2rank_probability','pdb_id'],
        ascending=[True,False,True]
    )
    .groupby('uniprot_id',group_keys=False)
    .head(5)
    .copy()
)

pilot_rows=[]
for pdb,g in tqdm(
    list(val_candidates.groupby('pdb_id')),
    desc='Centroid validation calculations'
):
    pilot_rows.extend(centroid_pdb_worker(pdb,g.to_dict('records')))

centroid_validation=pd.DataFrame(pilot_rows).merge(
    pockets[['pdb_id','uniprot_id','receptor_chain','pocket_residues_pdb']],
    on=['pdb_id','uniprot_id','receptor_chain'],how='left'
)

assert len(centroid_validation)==121,(
    f'Validation sample size changed: {len(centroid_validation)}. '
    'This may be valid after a dataset update, but review before proceeding.'
)
print('Validation cases:',len(centroid_validation))
print('Receptors covered:',centroid_validation['uniprot_id'].nunique())

### 12.1 Preserve the manual PyMOL decisions

These are explicit audit records from the completed validation session. They are not generated by the algorithm.

In [ ]:
CENTROID_MANUAL_PASS=pd.DataFrame([
    ['1KV6','B','Initial manual sanity check; centroid visually within selected pocket.'],
    ['1MVC','A','Initial manual sanity check; centroid visually within selected pocket.'],
    ['1NRL','A','Initial manual sanity check; centroid visually within selected pocket.'],
    ['1NRL','B','Initial manual sanity check; centroid visually within selected pocket.'],
    ['1P8D','A','Initial manual sanity check; centroid visually within selected pocket.'],
    ['1YUC','B','Automated QC outlier manually inspected; centroid within selected pocket.'],
    ['2O9I','A','Automated QC outlier manually inspected; centroid within selected pocket.'],
    ['2O9I','B','Automated QC outlier manually inspected; centroid within selected pocket.'],
    ['4DOR','B','Automated QC outlier manually inspected; centroid within selected pocket.'],
    ['5YXB','A','Automated QC outlier manually inspected; centroid within selected pocket.']
],columns=['pdb_id','receptor_chain','manual_note'])

### 12.2 Generate a scripted PyMOL measurement job

The script loads each validation structure in PyMOL and measures:
- nearest selected-pocket Cα distance;
- number/fraction of pocket Cα atoms within 6 Å and 8 Å;
- distance from the weighted centroid to the simple pocket Cα center;
- nearest organic-ligand atom distance (supporting evidence only).

The ligand is **not** required for PASS because apo structures are valid.

In [ ]:
cases=[]
for _,r in centroid_validation.iterrows():
    if r['status']!='PASS':
        continue
    cases.append({
        'pdb_id':str(r['pdb_id']),
        'receptor':str(r['receptor']),
        'uniprot_id':str(r['uniprot_id']),
        'chain':str(r['receptor_chain']),
        'centroid_x':float(r['centroid_x']),
        'centroid_y':float(r['centroid_y']),
        'centroid_z':float(r['centroid_z']),
        'pocket_residues':str(r['pocket_residues_pdb'])
    })

cases_path=CENTROID_VAL/'centroid_validation_cases.json'
cases_path.write_text(json.dumps(cases,indent=2),encoding='utf-8')

cases_file=str(cases_path.resolve()).replace('\\','/')
structures_dir=str(STRUCTURES.resolve()).replace('\\','/')
output_file=str((PROCESSED/'centroid_validation_metrics.csv').resolve()).replace('\\','/')

pymol_script=r"""
from pymol import cmd
import json,csv,math,os

CASES_FILE="__CASES_FILE__"
STRUCTURES="__STRUCTURES__"
OUTFILE="__OUTFILE__"

with open(CASES_FILE,encoding="utf-8") as f:
    cases=json.load(f)

def dist(a,b):
    return math.sqrt(sum((a[i]-b[i])**2 for i in range(3)))

rows=[]
for i,c in enumerate(cases,1):
    pdb,chain=c["pdb_id"],c["chain"]
    base={
        "pdb_id":pdb,"receptor":c["receptor"],
        "uniprot_id":c["uniprot_id"],"receptor_chain":chain
    }
    try:
        cmd.delete("all")
        cmd.load(os.path.join(STRUCTURES,pdb+".cif"),"mol")
        rec=f"mol and polymer.protein and chain {chain}"
        residues=[x.strip() for x in c["pocket_residues"].split(";") if x.strip()]
        pocket=f"({rec}) and resi {'+'.join(residues)}"
        centroid=(float(c["centroid_x"]),float(c["centroid_y"]),float(c["centroid_z"]))

        ca=[tuple(a.coord) for a in cmd.get_model(f"({pocket}) and name CA").atom]
        if not ca:
            raise ValueError("no pocket CA atoms found")

        d=[dist(centroid,x) for x in ca]
        pc=tuple(sum(x[j] for x in ca)/len(ca) for j in range(3))

        lig=[tuple(a.coord) for a in cmd.get_model("mol and organic and not solvent").atom]
        ld=[dist(centroid,x) for x in lig]

        rows.append({
            **base,
            "n_pocket_ca":len(ca),
            "nearest_pocket_ca_A":min(d),
            "pocket_ca_within6":sum(x<=6 for x in d),
            "pocket_ca_within8":sum(x<=8 for x in d),
            "centroid_to_pocket_center_A":dist(centroid,pc),
            "nearest_organic_ligand_A":min(ld) if ld else float("nan"),
            "ligand_atoms_within10":sum(x<=10 for x in ld),
            "measurement_status":"PASS","error":""
        })
    except Exception as e:
        rows.append({**base,"measurement_status":"ERROR","error":str(e)})
    print(f"[{i}/{len(cases)}] {pdb} chain {chain}")

fields=[
    "pdb_id","receptor","uniprot_id","receptor_chain","n_pocket_ca",
    "nearest_pocket_ca_A","pocket_ca_within6","pocket_ca_within8",
    "centroid_to_pocket_center_A","nearest_organic_ligand_A",
    "ligand_atoms_within10","measurement_status","error"
]
with open(OUTFILE,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=fields)
    w.writeheader(); w.writerows(rows)
print("saved:",OUTFILE)
"""

pymol_script=(
    pymol_script
    .replace('__CASES_FILE__',cases_file)
    .replace('__STRUCTURES__',structures_dir)
    .replace('__OUTFILE__',output_file)
)
script_path=CENTROID_VAL/'automated_centroid_validation.py'
script_path.write_text(pymol_script,encoding='utf-8')

print('Cases JSON:',cases_path)
print('Run this script inside PyMOL:',script_path)

After running `automated_centroid_validation.py` in PyMOL, continue with the QC cell below.

In [ ]:
metrics_path=PROCESSED/'centroid_validation_metrics.csv'
assert metrics_path.exists(),(
    'Run validation/centroid/automated_centroid_validation.py inside PyMOL first.'
)
metrics=pd.read_csv(metrics_path)
assert (metrics['measurement_status']=='PASS').all()

metrics['fraction_within6']=metrics['pocket_ca_within6']/metrics['n_pocket_ca']
metrics['fraction_within8']=metrics['pocket_ca_within8']/metrics['n_pocket_ca']

def upper_iqr_cutoff(s,k=3):
    s=s.dropna()
    q1,q3=s.quantile([0.25,0.75])
    return q3+k*(q3-q1)

nearest_cut=upper_iqr_cutoff(metrics['nearest_pocket_ca_A'])
center_cut=upper_iqr_cutoff(metrics['centroid_to_pocket_center_A'])
neighbor_cut=metrics['fraction_within8'].quantile(0.05)

metrics['qc_far_nearest']=metrics['nearest_pocket_ca_A']>nearest_cut
metrics['qc_far_center']=metrics['centroid_to_pocket_center_A']>center_cut
metrics['qc_low_neighbors']=metrics['fraction_within8']<neighbor_cut
metrics['auto_status']=np.where(
    metrics[['qc_far_nearest','qc_far_center','qc_low_neighbors']].any(axis=1),
    'REVIEW','PASS'
)

manual_keys=set(zip(CENTROID_MANUAL_PASS['pdb_id'],CENTROID_MANUAL_PASS['receptor_chain']))
metrics['manual_pass']=metrics.apply(
    lambda r:(r['pdb_id'],r['receptor_chain']) in manual_keys,axis=1
)
metrics['final_validation_status']=np.where(
    (metrics['auto_status']=='PASS')|metrics['manual_pass'],'PASS','REVIEW'
)

print(metrics['auto_status'].value_counts())
print(metrics['final_validation_status'].value_counts())
review=metrics[metrics['final_validation_status']!='PASS']
display(review)

assert review.empty,(
    'Centroid validation has unresolved REVIEW cases. Inspect them in PyMOL '
    'and add confirmed passes to CENTROID_MANUAL_PASS.'
)
metrics.to_csv(PROCESSED/'centroid_validation_metrics_final.csv',index=False)
print('Centroid validation gate PASS:',len(metrics),'/',len(metrics))

## 13. Scale the validated centroid calculation to all geometry units

In [ ]:
# Full batch
groups={
    pdb:g.to_dict('records')
    for pdb,g in pockets.groupby('pdb_id',sort=False)
}

centroid_rows=[]
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futs={ex.submit(centroid_pdb_worker,pdb,rows):pdb for pdb,rows in groups.items()}
    for fut in tqdm(as_completed(futs),total=len(futs),desc='Weighted centroids'):
        try:
            centroid_rows.extend(fut.result())
        except Exception as e:
            pdb=futs[fut]
            centroid_rows.append({'pdb_id':pdb,'status':'ERROR','error':str(e)})

centroids=pd.DataFrame(centroid_rows)

KEYS=['pdb_id','uniprot_id','receptor_chain']
assert len(centroids)==len(pockets)
assert not centroids.duplicated(KEYS).any()

centroids['use_fraction']=centroids['n_used']/centroids['n_requested']

print(centroids['status'].value_counts(dropna=False))

## 14. Full-batch centroid QC

Automatic QC flags:
- <80% of requested canonical pocket residues used;
- centroid-to-P2Rank-center shift above `Q3 + 3×IQR`.

The shift is only a QC signal because the SASA-weighted centroid and the P2Rank center are different definitions. Four completed-run shift outliers were manually inspected in PyMOL and accepted.

In [ ]:
good=centroids['status']=='PASS'
x=centroids.loc[good,'centroid_shift_from_p2rank_A'].dropna()
q1,q3=x.quantile([0.25,0.75])
shift_cutoff=q3+3*(q3-q1)

centroids['qc_low_use']=good & (centroids['use_fraction']<0.80)
centroids['qc_extreme_shift']=good & (
    centroids['centroid_shift_from_p2rank_A']>shift_cutoff
)
centroids['qc_status']=np.where(
    centroids['status']!='PASS','ERROR',
    np.where(centroids['qc_low_use']|centroids['qc_extreme_shift'],'REVIEW','PASS')
)

FULL_BATCH_MANUAL_PASS=pd.DataFrame([
    ['1MV9','A'],['6STI','A'],['6WMS','A'],['7WMO','A']
],columns=['pdb_id','receptor_chain'])

manual_keys=set(zip(
    FULL_BATCH_MANUAL_PASS['pdb_id'],
    FULL_BATCH_MANUAL_PASS['receptor_chain']
))
centroids['manual_qc']=centroids.apply(
    lambda r:'PASS_AFTER_VISUAL_REVIEW'
    if (r['pdb_id'],r['receptor_chain']) in manual_keys else '',
    axis=1
)
centroids['final_qc_status']=np.where(
    (centroids['qc_status']=='PASS')|
    (centroids['manual_qc']=='PASS_AFTER_VISUAL_REVIEW'),
    'PASS','REVIEW'
)

print('Shift cutoff:',shift_cutoff)
print('Automatic QC:')
print(centroids['qc_status'].value_counts())
print('Final QC:')
print(centroids['final_qc_status'].value_counts())

assert (centroids['final_qc_status']=='PASS').all()

centroids.to_csv(PROCESSED/'weighted_pocket_centroids.csv',index=False)
centroids.to_parquet(PROCESSED/'weighted_pocket_centroids.parquet',index=False)
centroids[centroids['qc_status']!='PASS'].to_csv(
    PROCESSED/'weighted_pocket_centroids_qc.csv',index=False
)
centroids[centroids['status']!='PASS'].to_csv(
    PROCESSED/'weighted_pocket_centroids_errors.csv',index=False
)

# Part III — H12 reference and PCA geometry

H12 is represented by:
- a canonical H12 residue range;
- a Cα-based PCA/SVD helical axis;
- a Pro Cα anchor where an analogous anchor is biologically available.

The automatic candidate builder uses mmCIF `_struct_conf` helix annotations mapped to canonical UniProt positions. It is a **candidate generator**, not an automatic biological truth.

## 15. Generate H12 structural candidates

In [ ]:
def _as_list(x):
    return x if isinstance(x,list) else [x]

def structure_helices_canonical(pdb,row):
    rec=row.get('receptor_original',row.get('receptor'))
    ch=row['receptor_chain']
    key=f"{pdb}|{rec}|{ch}"
    mp=numbering_maps.get(key,{})
    if not mp:
        return []

    d=MMCIF2Dict(str(STRUCTURES/f'{pdb}.cif'))
    req=[
        '_struct_conf.conf_type_id',
        '_struct_conf.beg_auth_asym_id',
        '_struct_conf.end_auth_asym_id',
        '_struct_conf.beg_auth_seq_id',
        '_struct_conf.end_auth_seq_id'
    ]
    if not all(k in d for k in req):
        return []

    vals=[_as_list(d[k]) for k in req]
    out=[]

    for typ,c1,c2,s1,s2 in zip(*vals):
        if not str(typ).startswith('HELX') or c1!=ch or c2!=ch:
            continue
        try:
            a,b=int(float(s1)),int(float(s2))
        except Exception:
            continue

        mapped=[]
        for p,c in mp.items():
            m=re.match(r'^(-?\d+)',str(p))
            if not m:
                continue
            n=int(m.group(1))
            if min(a,b)<=n<=max(a,b):
                mapped.append(int(c))

        if len(mapped)>=4:
            out.append({
                'pdb_id':pdb,
                'uniprot_id':row['uniprot_id'],
                'receptor':row['receptor'],
                'receptor_chain':ch,
                'canon_start':min(mapped),
                'canon_end':max(mapped),
                'n_mapped':len(mapped)
            })

    return out

h12_obs=[]
for _,r in tqdm(pockets.iterrows(),total=len(pockets),desc='H12 structural candidates'):
    hs=structure_helices_canonical(r['pdb_id'],r)
    if hs:
        # candidate H12 = most C-terminal annotated helix in this receptor chain
        h12_obs.append(max(hs,key=lambda x:x['canon_end']))

h12_obs=pd.DataFrame(h12_obs)
h12_obs.to_csv(PROCESSED/'h12_structure_candidates.csv',index=False)

# consensus proposal by receptor
cand=[]
for uid,g in h12_obs.groupby('uniprot_id'):
    start=int(round(g['canon_start'].median()))
    end=int(round(g['canon_end'].median()))
    seq=seqs.get(uid,'')

    # candidate Pro anchor: nearest Pro at/just upstream of H12 start
    pro_candidates=[
        i+1 for i,a in enumerate(seq)
        if a=='P' and start-20<=i+1<=start+2
    ]
    pro=max([p for p in pro_candidates if p<=start],default=np.nan)

    cand.append({
        'uniprot_id':uid,
        'receptor':UID_TO_NAME.get(uid,uid),
        'h12_start_candidate':start,
        'h12_end_candidate':end,
        'pro_anchor_candidate':pro,
        'n_structures_supporting':len(g),
        'source':'mmCIF _struct_conf + canonical mapping',
        'approved':'',
        'review_note':''
    })

h12_ref_candidates=pd.DataFrame(cand)
h12_ref_candidates.to_csv(
    PROCESSED/'h12_reference_candidates.csv',index=False
)

display(h12_ref_candidates)

## 16. Build tolerant H12 consensus and flag special cases

In [ ]:
h12_obs['helix_len']=h12_obs['canon_end']-h12_obs['canon_start']+1

h12_patterns=(
    h12_obs
    .groupby(['uniprot_id','receptor','canon_start','canon_end'])
    .agg(n_chain_obs=('pdb_id','size'),n_unique_pdb=('pdb_id','nunique'))
    .reset_index()
)

def nearest_pro(uid,start,window=6):
    seq=seqs.get(uid,'')
    lo=max(1,int(start)-window)
    hi=min(len(seq),int(start)+window)
    pros=[i for i in range(lo,hi+1) if seq[i-1]=='P']
    return min(pros,key=lambda p:abs(p-int(start))) if pros else np.nan

def tolerant_support(uid,start,end,tol=2):
    g=h12_obs[h12_obs['uniprot_id']==uid]
    if g.empty:
        return np.nan
    ok=(g['canon_start'].sub(start).abs()<=tol)&(g['canon_end'].sub(end).abs()<=tol)
    return ok.mean()

h12_ref=h12_ref_candidates.copy()
h12_ref['pro_anchor']=h12_ref.apply(
    lambda r:nearest_pro(r['uniprot_id'],r['h12_start_candidate']),axis=1
)
h12_ref['tolerant_support']=h12_ref.apply(
    lambda r:tolerant_support(
        r['uniprot_id'],r['h12_start_candidate'],r['h12_end_candidate'],tol=2
    ),axis=1
)
h12_ref=h12_ref.rename(columns={
    'h12_start_candidate':'h12_start',
    'h12_end_candidate':'h12_end'
})

# Documented biological/QC exceptions from the review session.
h12_ref['h12_applicable']=True
h12_ref['special_note']=''

# REV-ERB terminal candidate is treated as not-applicable for canonical H12 geometry.
m=h12_ref['uniprot_id'].isin(['P20393','Q14995'])
h12_ref.loc[m,'h12_applicable']=False
h12_ref.loc[m,'special_note']=(
    'Documented exception: canonical H12 geometry not applied; '
    'terminal-helix candidate is retained only for audit.'
)

# PR reviewed range/anchor.
m=h12_ref['uniprot_id'].eq('P06401')
h12_ref.loc[m,'h12_start']=909
h12_ref.loc[m,'h12_end']=921
h12_ref.loc[m,'pro_anchor']=906
h12_ref.loc[m,'special_note']=(
    'Manual reference review: H12 909-921; Pro906 anchor.'
)

# HNF4alpha: keep structural H12 candidate, but no equivalent Pro anchor was accepted.
m=h12_ref['uniprot_id'].eq('P41235')
h12_ref.loc[m,'h12_start']=367
h12_ref.loc[m,'h12_end']=375
h12_ref.loc[m,'pro_anchor']=np.nan
h12_ref.loc[m,'special_note']=(
    'Manual reference review: structural H12 candidate 367-375; '
    'no equivalent Pro anchor accepted. Supervisor confirmation recommended.'
)

h12_ref['reference_status']=np.where(
    h12_ref['h12_applicable'],
    'READY_FOR_PYMOL_VALIDATION',
    'NOT_APPLICABLE'
)

h12_ref.to_csv(PROCESSED/'h12_reference_reviewed.csv',index=False)
display(h12_ref[[
    'uniprot_id','receptor','h12_start','h12_end','pro_anchor',
    'tolerant_support','h12_applicable','reference_status','special_note'
]])

## 17. H12 reference gate

All H12-applicable receptors must have a canonical start/end range before pilot geometry can be calculated. A Pro anchor is allowed to be missing only as an explicitly documented exception (currently HNF4α).

REV-ERB rows are retained in the reference audit but excluded from H12 PCA geometry.

In [ ]:
geometry_uids=set(pockets['uniprot_id'])
applicable=set(h12_ref.loc[h12_ref['h12_applicable'],'uniprot_id'])
not_applicable=set(h12_ref.loc[~h12_ref['h12_applicable'],'uniprot_id'])

missing=geometry_uids-(applicable|not_applicable)
assert not missing,f'Missing H12 reference decisions: {sorted(missing)}'

bad_ranges=h12_ref[
    h12_ref['h12_applicable']&
    (h12_ref['h12_start'].isna()|h12_ref['h12_end'].isna())
]
assert bad_ranges.empty

print('H12-applicable receptor types:',len(applicable))
print('Documented not-applicable receptor types:',sorted(not_applicable))
print('Reference gate PASS')

## 18. H12 PCA geometry function

For each applicable receptor:
1. map the canonical H12 range back to deposited coordinates;
2. collect H12 Cα atoms;
3. fit the long helical axis by SVD/PCA;
4. orient the axis consistently from N- to C-terminus;
5. return the Pro Cα coordinate when an accepted anchor exists.

A missing Pro anchor does not invalidate the H12 axis; it is recorded explicitly.

In [ ]:
h12_ref_idx=h12_ref.set_index('uniprot_id').to_dict('index')

def find_residue_by_label(chain,label):
    for r in chain:
        if r.id[0]==' ' and residue_label(r)==str(label):
            return r
    return None

def h12_for_chain(st,row):
    pdb=row['pdb_id']
    uid=row['uniprot_id']
    ch=row['receptor_chain']
    rec_orig=row.get('receptor_original',row.get('receptor'))

    base={
        'pdb_id':pdb,'uniprot_id':uid,'receptor':row['receptor'],
        'receptor_chain':ch
    }

    try:
        ref=h12_ref_idx[uid]

        if not bool(ref['h12_applicable']):
            return {
                **base,'status':'NOT_APPLICABLE_NO_H12','error':'',
                'h12_start':np.nan,'h12_end':np.nan
            }

        start,end=int(ref['h12_start']),int(ref['h12_end'])
        pro_raw=ref['pro_anchor']

        key=f'{pdb}|{rec_orig}|{ch}'
        mp=numbering_maps.get(key,{})
        c2p={int(v):str(k) for k,v in mp.items()}

        if ch not in st[0]:
            raise ValueError('receptor chain missing')
        chain=st[0][ch]

        coords=[]
        canon_used=[]
        for c in range(start,end+1):
            p=c2p.get(c)
            if p is None:
                continue
            r=find_residue_by_label(chain,p)
            if r is None or 'CA' not in r:
                continue
            coords.append(r['CA'].get_coord().astype(float))
            canon_used.append(c)

        if len(coords)<4:
            raise ValueError('fewer than 4 H12 C-alpha coordinates')

        X=np.asarray(coords,float)
        X0=X-X.mean(axis=0)
        _,_,vt=np.linalg.svd(X0,full_matrices=False)
        axis=vt[0]/np.linalg.norm(vt[0])

        direction=X[-1]-X[0]
        if np.dot(axis,direction)<0:
            axis=-axis

        pro_ca=np.array([np.nan,np.nan,np.nan],dtype=float)
        pro_canonical=np.nan
        if pd.notna(pro_raw):
            pro_canonical=int(pro_raw)
            pro_pdb=c2p.get(pro_canonical)
            if pro_pdb is not None:
                pr=find_residue_by_label(chain,pro_pdb)
                if pr is not None and 'CA' in pr:
                    pro_ca=pr['CA'].get_coord().astype(float)

        return {
            **base,
            'h12_start':start,'h12_end':end,
            'h12_canonical_used':';'.join(map(str,canon_used)),
            'n_h12_ca':len(canon_used),
            'pro_anchor_canonical':pro_canonical,
            'pro_x':float(pro_ca[0]),'pro_y':float(pro_ca[1]),'pro_z':float(pro_ca[2]),
            'h12_axis_x':float(axis[0]),'h12_axis_y':float(axis[1]),'h12_axis_z':float(axis[2]),
            'status':'PASS','error':''
        }
    except Exception as e:
        return {**base,'status':'ERROR','error':str(e)}

def h12_pdb_worker(pdb,rows):
    parser=MMCIFParser(QUIET=True,auth_chains=True,auth_residues=True)
    try:
        st=parser.get_structure(pdb,STRUCTURES/f'{pdb}.cif')
        return [h12_for_chain(st,row) for row in rows]
    except Exception as e:
        return [{
            'pdb_id':pdb,'uniprot_id':row.get('uniprot_id'),
            'receptor':row.get('receptor'),'receptor_chain':row.get('receptor_chain'),
            'status':'ERROR','error':str(e)
        } for row in rows]

## 19. H12 validation sample — **next required step**

Select up to five structures per H12-applicable receptor, calculate their PCA axes, and validate them in PyMOL before scaling. This section intentionally does **not** auto-mark validation as complete.

For reporting: H12 reference construction is prepared, but the full H12 validation/full batch should be described as ongoing until this gate passes.

In [ ]:
h12_candidates=(
    pockets[pockets['uniprot_id'].isin(applicable)]
    .sort_values(['uniprot_id','pdb_id'])
    .groupby('uniprot_id',group_keys=False)
    .head(5)
    .copy()
)

h12_pilot=[]
for pdb,g in tqdm(
    list(h12_candidates.groupby('pdb_id')),
    desc='H12 pilot calculations'
):
    h12_pilot.extend(h12_pdb_worker(pdb,g.to_dict('records')))

h12_validation=pd.DataFrame(h12_pilot)
h12_validation.to_csv(PROCESSED/'h12_validation_queue.csv',index=False)

print(h12_validation['status'].value_counts(dropna=False))
print('Pilot cases:',len(h12_validation))
print('Receptors covered:',h12_validation['uniprot_id'].nunique())
display(h12_validation.head())

## 20. H12 full batch — run only after the H12 validation gate is completed

The code is provided now for reproducibility, but it should only be executed after the pilot PyMOL validation has been documented.

In [ ]:
H12_VALIDATION_CONFIRMED=False  # set True only after documented PyMOL validation

assert H12_VALIDATION_CONFIRMED,(
    'STOP: complete and document H12 PyMOL validation before scaling.'
)

groups={
    pdb:g.to_dict('records')
    for pdb,g in pockets.groupby('pdb_id',sort=False)
}

h12_rows=[]
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futs={ex.submit(h12_pdb_worker,pdb,rows):pdb for pdb,rows in groups.items()}
    for fut in tqdm(as_completed(futs),total=len(futs),desc='H12 geometry'):
        try:
            h12_rows.extend(fut.result())
        except Exception as e:
            pdb=futs[fut]
            h12_rows.append({'pdb_id':pdb,'status':'ERROR','error':str(e)})

h12=pd.DataFrame(h12_rows)
h12.to_csv(PROCESSED/'h12_geometry.csv',index=False)
h12.to_parquet(PROCESSED/'h12_geometry.parquet',index=False)
h12[h12['status']=='ERROR'].to_csv(
    PROCESSED/'h12_geometry_errors.csv',index=False
)

print(h12['status'].value_counts(dropna=False))

# Final handoff and status

### Completed
- Student A geometry prerequisite audit and canonical numbering.
- P2Rank batch and chain-aware pocket selection.
- Canonical pocket residue tables.
- Weighted centroid implementation.
- 121-case centroid validation across 28 receptors.
- Full 1,167-unit centroid batch and QC.

### Ongoing
- H12 reference confirmation for special cases.
- H12 pilot validation in PyMOL.
- Full H12 scaling and final combined Student B Parquet.

### Files downstream students need after H12 is complete
- `canonical_numbering_maps_v2.json`
- `geometry_structures_handoff_clean.csv`
- `p2rank_pockets_final_named.parquet`
- `weighted_pocket_centroids.parquet`
- `h12_geometry.parquet`
- validation/QC tables